In [ ]:
import earthkit.data
import earthkit.plots

# Polytope Example for Trajectory
## Using Ensemble Prediction System (ENS) Data from FDB

<div style="border:2px solid #f39c12; background-color:#fcf8e3; padding:10px; border-radius:5px;">
  <strong>⚠️ Warning:</strong> Operational FDB only retains data for 2 days in the past.
</div>

### Polytope Trajectory Feature [Documentation](https://polytope.readthedocs.io/en/latest/Service/Features/trajectory/)

### Shipping Forecast: 10m wind along a North Sea ferry route

Ferry and shipping operators need wind conditions along their route, not just at the departure and arrival ports, since conditions can change considerably over open water. Here we extract the **10m wind** vector along a series of waypoints on a North Sea ferry route from Rotterdam to Bergen, inflating a 0.3° corridor around the route to also capture nearby grid points.

In [ ]:
# Approximate waypoints along a North Sea ferry route (lat, lon)
waypoints = [
    [51.9, 4.5],    # Rotterdam
    [54.5, 4.0],    # North Sea
    [60.4, 5.3],    # Bergen
]

request = {
    "class": "od",
    "stream": "enfo",
    "type": "pf",
    "date": "-1",
    "time": "0000",
    "expver": "0001",
    "domain": "g",
    "levtype": "sfc",
    "param": "165/166",  # 10m u and v wind components
    "number": "1",
    "step": "0",
    "feature": {
        "type": "trajectory",
        "points": waypoints,
        "inflation" : [0, 0.3],
        "inflate" : 'box',
        "axes": ["latitude", "longitude"],
    },
}
data = earthkit.data.from_source(
    "polytope", "ecmwf-mars", request, address="polytope.ecmwf.int", stream=False)

In [ ]:
dx = data.to_xarray()
dx

In [ ]:
import numpy as np

dx["wind_speed"] = np.sqrt(dx["10u"] ** 2 + dx["10v"] ** 2)
dx["wind_speed"].attrs["units"] = "m/s"
dx["wind_speed"]

In [ ]:
chart = earthkit.plots.Map(domain=[0, 12, 50, 62])
chart.point_cloud(
    dx,
    x="longitude",
    y="latitude",
    z="wind_speed",
    units="m/s",
    # s=20,
)
chart.cities(
    medium_cities_kwargs={"text": {"color": "red"}},
)

chart.coastlines()
chart.gridlines()
chart.legend()
chart.title("10m wind speed along the Rotterdam\u2013Bergen ferry route")
chart.show()